<img src="https://raw.githubusercontent.com/MohammedAly22/qwencleo-asr/main/assets/QwenCleo-ASR-Banner.png" width="100%"/>

# 🎙️ QwenCleo-ASR — Chunked Long-Audio Transcription

Egyptian Arabic & code-switching speech recognition, built on Qwen3-ASR.

[Model](https://huggingface.co/mohammedaly22/QwenCleo-ASR) · [GitHub](https://github.com/MohammedAly22/qwencleo-asr) · [PyPI](https://pypi.org/project/qwencleo-asr/)

> **Runtime → Change runtime type → GPU** before running. Then run the cells top to bottom.

Split long or live audio into overlapping windows and transcribe each — convenient for captioning without a server.

> ℹ️ This is **chunked transcription**, not true streaming. For token-by-token streaming see the **vLLM** notebook.

## 1. Install

In [ ]:
# Install QwenCleo-ASR (Colab already has a CUDA-matched torch)
!pip install -q qwencleo-asr

## 2. Sample audio

In [ ]:
# Grab a sample Egyptian/code-switching clip to transcribe
import urllib.request, os
URL = 'https://huggingface.co/mohammedaly22/QwenCleo-ASR/resolve/main/assets/sample.wav'
FALLBACK = 'https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen3-ASR-Repo/asr_en.wav'
path = 'sample.wav'
try:
    urllib.request.urlretrieve(URL, path)
except Exception:
    urllib.request.urlretrieve(FALLBACK, path)
print('saved', path, os.path.getsize(path), 'bytes')

## 3. Stream a file window by window

In [ ]:
from IPython.display import display, Audio
from qwencleo_asr import QwenCleoASR, stream_file

audio_path = "sample.wav"
display(Audio(audio_path, autoplay=False))

asr = QwenCleoASR()
for chunk in stream_file(asr, audio_path, chunk_s=20, overlap_s=2):
    tag = ' (final)' if chunk.is_final else ''
    print(f'[{chunk.start:.0f}-{chunk.end:.0f}s]{tag} {chunk.text}')

## 4. Mic-style frame feeding (ChunkedSession)

Simulates feeding audio frames (e.g. from a microphone) and pulling transcripts as enough audio accumulates.

In [ ]:
import soundfile as sf
from qwencleo_asr import QwenCleoASR, ChunkedSession
from IPython.display import display, Audio


display(Audio(audio_path, autoplay=False))

asr = QwenCleoASR()
audio, sr = sf.read('sample.wav', dtype='float32')
if audio.ndim > 1:
    audio = audio.mean(axis=1)

sess = ChunkedSession(asr, sr=sr, chunk_s=8)
# feed in 1-second frames
for i in range(0, len(audio), sr):
    frame = audio[i:i+sr]
    for chunk in sess.add(frame):
        print(chunk.text)
for chunk in sess.flush():
    print(chunk.text)